# This notebook launches a script, meant for the scriptQueue, but via a notebook.
## This is required to debug scripts

In [2]:
import sys
import asyncio
import time

import numpy as np
import logging 
import yaml

from lsst.ts import salobj

from lsst.ts.idl.enums.Script import ScriptState

In [1]:
import astropy

In [3]:
from lsst.ts.externalscripts.auxtel.latiss_acquire_and_take_sequence import LatissAcquireAndTakeSequence

In [4]:
stream_handler = logging.StreamHandler(sys.stdout)
# if you want logging
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

# turn off logging for matplotlib
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)

In [ ]:
# Note that you need to restart here if the script completes

In [5]:
script = LatissAcquireAndTakeSequence(index=1)  # this essentially calls the init method

DEBUG:Script.ATCS:atmcs: Adding all resources.
atmcs: Adding all resources.
DEBUG:Script.ATCS:atptg: Adding all resources.
atptg: Adding all resources.
DEBUG:Script.ATCS:ataos: Adding all resources.
ataos: Adding all resources.
DEBUG:Script.ATCS:atpneumatics: Adding all resources.
atpneumatics: Adding all resources.
DEBUG:Script.ATCS:athexapod: Adding all resources.
athexapod: Adding all resources.
DEBUG:Script.ATCS:atdome: Adding all resources.
atdome: Adding all resources.
DEBUG:Script.ATCS:atdometrajectory: Adding all resources.
atdometrajectory: Adding all resources.
DEBUG:Script.LATISS:atcamera: Adding all resources.
atcamera: Adding all resources.
DEBUG:Script.LATISS:atspectrograph: Adding all resources.
atspectrograph: Adding all resources.
DEBUG:Script.LATISS:atheaderservice: Adding all resources.
atheaderservice: Adding all resources.
DEBUG:Script.LATISS:atarchiver: Adding all resources.
atarchiver: Adding all resources.
INFO:ATDomeTrajectory:Read historical data in 0.01 sec
R

In [6]:
# make sure all remotes etc are running
await script.start_task

In [7]:
# set wrap strategy
script.atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

logMessage DDS read queue is filling: 14 of 100 elements
logMessage DDS read queue is filling: 10 of 100 elements


# Emulate how the scriptQueue launches scripts

In [ ]:
# Needed to run the script a 2nd time 
script.set_state(ScriptState.UNCONFIGURED)

In [ ]:
# configuration = yaml.safe_dump({"object_name": 'HIP 31636',
#                                 "do_acquire": False,
#                                 "acq_filter" : 'BG40',
#                                 "acq_grating" : 'empty_1',
#                                 "exposure_time_sequence" : [4, 4, 1, 2, 1, 2], 
#                                 "filter_sequence": ['empty_1','quadnotch1','BG40', 'BG40','RG610','RG610'], 
#                                 "grating_sequence": ['ronchi90lpmm','ronchi90lpmm','empty_1','empty_1','empty_1','empty_1'],
#                                 "doPointingModel": False,
#                                 "dataPath": '/project/shared/auxTel/rerun/quickLook',
#                                 "max_acq_iter": 0 })
# print(configuration)

In [ ]:
# configuration = yaml.safe_dump({"object_name": 'HIP 31636',
#                                 "do_acquire": False,
#                                 "acq_filter" : 'BG40',
#                                 "acq_grating" : 'empty_1',
#                                 "exposure_time_sequence" : [4, 4, 2, 2], 
#                                 "filter_sequence": ['empty_1','quadnotch1', 'BG40','RG610'], 
#                                 "grating_sequence": ['ronchi90lpmm','ronchi90lpmm','empty_1','empty_1'],
#                                 "doPointingModel": False,
#                                 "dataPath": '/project/shared/auxTel/rerun/quickLook',
#                                 "max_acq_iter": 0 })
# print(configuration)

In [ ]:
# configuration = yaml.safe_dump({"object_name": 'HIP 46562',
#                                 "do_acquire": True,
#                                 "acq_filter" : 'BG40',
#                                 "acq_grating" : 'empty_1',
#                                 "exposure_time_sequence" : [4, 4, 2, 2], 
#                                 "filter_sequence": ['empty_1','quadnotch1', 'BG40','RG610'], 
#                                 "grating_sequence": ['ronchi90lpmm','ronchi90lpmm','empty_1','empty_1'],
#                                 "doPointingModel": False,
#                                 "dataPath": '/project/shared/auxTel/rerun/quickLook',
#                                 "max_acq_iter": 1,
#                                 "target_pointing_verification": False})
# print(configuration)

In [ ]:
configuration = yaml.safe_dump({"object_name": 'HD24661',
                                "do_acquire": True,
                                "acq_filter" : 'RG610',
                                "acq_grating" : 'empty_1',
                                "exposure_time_sequence" : [4, 4, 2, 2], 
                                "filter_sequence": ['empty_1','quadnotch1', 'BG40','RG610'], 
                                "grating_sequence": ['ronchi90lpmm','ronchi90lpmm','empty_1','empty_1'],
                                "doPointingModel": False,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook',
                                "max_acq_iter": 4,
                                "target_pointing_verification": True})
print(configuration)

In [ ]:
config_data = script.cmd_configure.DataType()
config_data.config = configuration
await script.do_configure(config_data)

In [ ]:
# turn on ataos
await script.atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [ ]:
tmp=await script.atcs.rem.ataos.evt_focusOffsetSummary.aget()
print(tmp)

In [ ]:
tmp=await script.atcs.rem.ataos.evt_pointingOffsetSummary.aget()
print(tmp)

In [ ]:
# run script
group_id_data = script.cmd_setGroupId.DataType(
                groupId=astropy.time.Time.now().isot
            )
await script.do_setGroupId(group_id_data)

run_data = script.cmd_run.DataType()
await script.do_run(run_data)
await script.done_task

In [ ]:
await script.atcs.stop_tracking()

In [ ]:
fig1 = plt.figure(1, figsize=(12,8))
ax11 = fig1.add_subplot(121)
ax11.set_title(f"intra - {script.intra_visit_id}")
ax11.imshow(script.I1[0].image0)
ax11.contour(script.algo.pMask) 
ax12 = fig1.add_subplot(122)
ax12.set_title(f"extra - {script.extra_visit_id}")
ax12.imshow(script.I2[0].image0)
ax12.contour(script.algo.pMask) 

In [ ]:
await script.attcs.stop_tracking()

In [ ]:
#await script.attcs.slew_object('HD 110589')

In [ ]:
target_list=["HD  88678"]
# Reverse the targets?
target_list=target_list[::-1]
print(target_list)

In [ ]:
for target in target_list:
    script.set_state(ScriptState.UNCONFIGURED)
    
    configuration = yaml.safe_dump({"object_name": target, 
                                "exposure_time": 2, 
                                "filter": 'empty_1', 
                                "grating": 'empty_1',
                                "doPointingModel": True,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook'})
    print(configuration)
    
    config_data = script.cmd_configure.DataType()
    config_data.config = configuration
    await script.do_configure(config_data)
    
    await script.run()

In [ ]:
print("Done")

In [ ]:
# stops the messaging output if you had to stop the above cell
await script.attcs.cancel_not_done(script.attcs.scheduled_coro)

In [ ]:
await script.attcs.atptg.cmd_stopTracking.start()

In [ ]:
await script.latiss.take_object(exptime=15)

In [ ]:
await script.attcs.offset_xy(x=45.1975, y=-48.6315)

In [ ]:
await script.latiss.take_engtest(exptime=10)